In [ ]:
import io
import chess.pgn
import pandas as pd
import numpy as np
import requests
import zstandard 

PIECE_VALUE = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9
}

def material(board, color):
    return sum(PIECE_VALUE.get(p.piece_type, 0)
        for p in board.piece_map().values() if p.color == color)

def extract_features(game, n_moves=10):
    h = game.headers
    base, inc = h['TimeControl'].split('+')
    board = game.board()
    clocks = {chess.WHITE: float('nan'), chess.BLACK: float('nan')}
    plies = 0
    last_node = None
    for node in game.mainline():
        mover = board.turn
        board.push(node.move)
        c = node.clock()
        if c is not None:
            clocks[mover] = c
        plies += 1
        last_node = node
        if plies == 2 * n_moves:
            break
    if plies < 2 * n_moves:
        return None
    if last_node.is_end():
        return None
    if board.is_game_over():
        return None
    w_elo, b_elo = int(h['WhiteElo']), int(h['BlackElo'])
    return {
        'game_id': h['Site'].split('/')[-1],
        'date': h['UTCDate'], 'time': h['UTCTime'],
        'white': h['White'], 'black': h['Black'], 
        'white_elo': w_elo, 'black_elo': b_elo,
        'elo_diff': w_elo - b_elo,
        'base_sec': int(base), 'inc_sec': int(inc),
        'material_diff': material(board, chess.WHITE) - material(board, chess.BLACK), 'white_clock': clocks[chess.WHITE],
        'black_clock': clocks[chess.BLACK], 'clock_diff': clocks[chess.WHITE] - clocks[chess.BLACK],
        'result': h['Result']
    }
def read_games(url, max_games=100, n_moves=10):
    resp = requests.get(url, stream=True)
    rows = []
    stream = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(resp.raw), encoding='utf-8')
    while len(rows) < max_games:
        game = chess.pgn.read_game(stream)
        if game is None:
            break
        try:
            features = extract_features(game, n_moves)
            if features is not None:
                rows.append(features)
        except (KeyError, ValueError):
            continue
    resp.close()
    return pd.DataFrame(rows)
        
        
df = read_games('https://database.lichess.org/standard/lichess_db_standard_rated_2017-08.pgn.zst', max_games=5000)
        

In [ ]:
df = df[df['result'].isin(['1-0', '0-1'])].copy()
df['white_won'] = (df['result'] == '1-0').astype(int)

In [8]:
df

,game_id,white_elo,black_elo,elo_diff,base_sec,inc_sec,material_diff,white_clock,black_clock,clock_diff,result
0,0KlFojax,1946,2089,-143,600,0,-3,540.0,545.0,-5.0,0-1
1,cgG7yWcC,2055,2008,47,60,1,0,64.0,52.0,12.0,1-0
2,m36DVKz1,1785,1738,47,180,0,-1,158.0,67.0,91.0,1-0
3,xvuIKSP6,1945,1867,78,600,0,0,555.0,570.0,-15.0,1-0
4,ztrFJwbI,1210,1342,-132,300,3,-1,291.0,240.0,51.0,0-1
...,...,...,...,...,...,...,...,...,...,...,...
4995,GOInFufV,1940,1970,-30,180,2,0,134.0,142.0,-8.0,0-1
4996,Ndh2K14h,1708,1714,-6,300,0,0,269.0,289.0,-20.0,0-1
4997,wO1RVPkj,2022,1858,164,180,2,-1,150.0,165.0,-15.0,0-1
4998,l9Gq6baP,1321,1304,17,180,2,0,157.0,163.0,-6.0,0-1
